# Async fan-out / fan-in with the OpenAI Responses API

This notebook demonstrates a common async LLM workflow:

1. Send several **independent analysis prompts** over the same document in parallel.
2. Collect those results.
3. Send one final **synthesis** prompt that combines them into an executive brief.

It uses the official Python SDK's `AsyncOpenAI` client and the Responses API, which OpenAI recommends for new projects. 

## What you'll need

- Python 3.9+
- `OPENAI_API_KEY` set in your environment
- The `openai` package installed

Example install command:

```bash
pip install -U openai nest_asyncio pandas
```


In [1]:
import os
import time
import asyncio
from textwrap import shorten

import nest_asyncio
from openai import OpenAI, AsyncOpenAI

nest_asyncio.apply()

if not os.getenv('OPENAI_API_KEY'):
    raise EnvironmentError('OPENAI_API_KEY is not set in the environment.')

print('API key detected.')


API key detected.


## Paste your source text here

Replace the placeholder below with the article, transcript, memo, or report you want to analyze.


In [12]:
from pathlib import Path

TEXT_FILE = "earnings_report.txt"   # change if needed

# Load the document
DOCUMENT_TEXT = Path(TEXT_FILE).read_text(encoding="utf-8")

print(f"Loaded {TEXT_FILE}")
print(f"Original length: {len(DOCUMENT_TEXT):,} characters")

# ---- Truncate for demo performance ----
MAX_CHARS = 40000

if len(DOCUMENT_TEXT) > MAX_CHARS:
    DOCUMENT_TEXT = DOCUMENT_TEXT[:MAX_CHARS]
    print(f"Truncated to: {len(DOCUMENT_TEXT):,} characters")
else:
    print("No truncation needed")

# Preview
print("\nPreview:")
print(DOCUMENT_TEXT[:500])

Loaded earnings_report.txt
Original length: 208,922 characters
Truncated to: 40,000 characters

Preview:
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, DC 20549

FORM 10-Q

☑ Quarterly report pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934
For the quarterly period ended September 30, 2025

or

☐ Transition report pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934
For the transition period from  __________ to __________
Commission file number 1-3950

Ford Motor Company
(Exact name of Registrant as specified in its charter)
Delaware		38-0549190


## Choose a model

You can swap this for another text-capable model available in your account.


In [13]:
MODEL = 'gpt-5-mini'


## Define the analysis prompts

These are the independent subtasks that can be run concurrently.


In [14]:
SUBTASK_PROMPTS = {
    'risks': (
        'Read the document and extract the top risks. '
        'Return a concise bulleted list with short explanations.'
    ),
    'opportunities': (
        'Read the document and extract the top opportunities. '
        'Return a concise bulleted list with short explanations.'
    ),
    'action_items': (
        'Read the document and extract the concrete action items. '
        'Return a concise bulleted list with owners or likely owners if inferable.'
    ),
    'summary': (
        'Write a five-bullet executive summary of the document for a busy stakeholder.'
    ),
}

list(SUBTASK_PROMPTS)


['risks', 'opportunities', 'action_items', 'summary']

## Helper functions

The notebook uses the Responses API for each subtask and then again for the final synthesis step. OpenAI documents the Responses API as the recommended path for new projects, and the Python SDK supports both sync and async clients. 

In [6]:
sync_client = OpenAI()
async_client = AsyncOpenAI()

SYSTEM_INSTRUCTIONS = (
    'You are a careful analyst. Be concise, specific, and avoid filler. '
    'Base your answer only on the provided document.'
)

def build_input(task_instruction: str, document_text: str) -> str:
    return (
        f'System instructions:\n{SYSTEM_INSTRUCTIONS}\n\n'
        f'Task:\n{task_instruction}\n\n'
        f'Document:\n{document_text}'
    )

def run_subtask_sync(task_name: str, task_instruction: str, document_text: str) -> str:
    response = sync_client.responses.create(
        model=MODEL,
        input=build_input(task_instruction, document_text),
    )
    return response.output_text.strip()

async def run_subtask_async(task_name: str, task_instruction: str, document_text: str) -> str:
    response = await async_client.responses.create(
        model=MODEL,
        input=build_input(task_instruction, document_text),
    )
    return response.output_text.strip()

def build_synthesis_prompt(results: dict[str, str]) -> str:
    return f'''
System instructions:
{SYSTEM_INSTRUCTIONS}

Task:
Combine the following analyses into a polished executive brief.

Your output should contain:
1. A one-paragraph overview
2. Top risks
3. Top opportunities
4. Recommended next actions

Use clear headings and concise bullets where helpful.

Inputs:

RISKS:
{results['risks']}

OPPORTUNITIES:
{results['opportunities']}

ACTION ITEMS:
{results['action_items']}

SUMMARY:
{results['summary']}
'''.strip()

def synthesize_sync(results: dict[str, str]) -> str:
    response = sync_client.responses.create(
        model=MODEL,
        input=build_synthesis_prompt(results),
    )
    return response.output_text.strip()

async def synthesize_async(results: dict[str, str]) -> str:
    response = await async_client.responses.create(
        model=MODEL,
        input=build_synthesis_prompt(results),
    )
    return response.output_text.strip()


## Sequential baseline

This runs the subtasks one after another, then performs the synthesis.


In [7]:
def run_pipeline_sync(document_text: str):
    start = time.perf_counter()
    results = {}

    for name, instruction in SUBTASK_PROMPTS.items():
        results[name] = run_subtask_sync(name, instruction, document_text)

    executive_brief = synthesize_sync(results)
    elapsed = time.perf_counter() - start
    return results, executive_brief, elapsed


## Async fan-out / fan-in version

This launches the independent subtasks concurrently with `asyncio.gather`, then performs a final synthesis request after all of them complete.


In [8]:
async def run_pipeline_async(document_text: str):
    start = time.perf_counter()

    tasks = [
        run_subtask_async(name, instruction, document_text)
        for name, instruction in SUBTASK_PROMPTS.items()
    ]

    outputs = await asyncio.gather(*tasks)
    results = dict(zip(SUBTASK_PROMPTS.keys(), outputs))

    executive_brief = await synthesize_async(results)
    elapsed = time.perf_counter() - start
    return results, executive_brief, elapsed


## Run both versions and compare timing


In [15]:
sync_results, sync_brief, sync_elapsed = run_pipeline_sync(DOCUMENT_TEXT)
async_results, async_brief, async_elapsed = asyncio.run(run_pipeline_async(DOCUMENT_TEXT))

print(f'Sequential pipeline: {sync_elapsed:.2f} seconds')
print(f'Async pipeline:      {async_elapsed:.2f} seconds')
print(f'Speedup:              {sync_elapsed / async_elapsed:.2f}x')


Sequential pipeline: 109.58 seconds
Async pipeline:      51.35 seconds
Speedup:              2.13x


## Inspect the intermediate parallel outputs


In [16]:
for key, value in async_results.items():
    print('=' * 80)
    print(key.upper())
    print('-' * 80)
    print(value)
    print()


RISKS
--------------------------------------------------------------------------------
- High leverage and refinancing/liquidity risk — Total liabilities of $253.6B vs total equity $47.4B; significant short‑ and long‑term debt balances (Ford Credit current debt ~$53.7B; Company excluding Ford Credit long‑term debt ~$17.9B). The company continues to rely on debt issuance (YTD proceeds ~$37.0B) and capital markets activity to fund operations and securitizations.

- Concentration and credit risk in Ford Credit — Finance receivables, net ~$108.4B with allowance for credit losses ~$897M; a material portion is securitized and restricted to pay securitization obligations (receivables not available to other creditors). Although past‑due rates are low (~1.3% consumer), gross charge‑offs remain meaningful (first nine months 2025 charge‑offs $490M).

- Revenue volatility from incentives, returns and deferred service contract obligations — Vehicle revenue is sensitive to marketing incentives and p

## Final executive brief


In [17]:
print(async_brief)


One-paragraph overview
Ford delivered stronger quarterly profitability in Q3 2025 (consolidated revenue $50.534B; operating income $1.558B; net income attributable $2.447B) against a mixed nine‑month performance (YTD operating income down to $2.388B). Balance‑sheet scale and liquidity remain material strengths (total assets $300.990B; cash + marketable securities ≈ $42.2B; operating cash flow YTD $17.4B), but the enterprise carries high leverage and material Ford Credit securitization funding, creating refinancing and liquidity sensitivity. Key drivers near‑term are Ford Credit receivables and funding structures, variable vehicle revenue and incentives, deferred extended‑service contract recognition, and volatile tax/deferred tax items and market/derivative mark‑to‑market impacts. Management has actionable disclosure and accounting standard adoptions to complete while prioritizing tax, securitization, and revenue‑estimate controls.

Top risks
- High leverage and refinancing/liquidity r

## Notes

- This pattern is useful when several prompts depend on the **same input**, but not on **each other**.
- The final synthesis step is still sequential because it depends on the outputs of the first stage.
- For larger workloads, add concurrency limits with a semaphore to avoid pushing into account rate limits.
- If you want machine-readable intermediate results, you can adapt the subtasks to return structured JSON using the Responses API. 
